# Isolation Forest для обнаружения аномалий в WMS

Данный ноутбук реализует обучение модели Isolation Forest для обнаружения аномалий в системе управления складом (WMS).

Модель обучается на трех наборах данных:
- **Сотрудники** (employee) - аномалии в поведении сотрудников
- **Маршруты** (route) - аномалии в маршрутах сборки
- **Товары** (item) - аномалии в характеристиках товаров

## Ячейка 1: Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
import warnings
from datetime import datetime
import os

warnings.filterwarnings('ignore')

# Определение окружения
IS_DOCKER = os.path.exists('/.dockerenv') or os.environ.get('DOCKER_CONTAINER') == '1'

# Настройка стиля графиков
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('Библиотеки успешно импортированы')

## Ячейка 2: Подключение к базе данных

In [ ]:
# Параметры подключения к PostgreSQL
DB_CONFIG = {
    'host': os.environ.get('DB_HOST', 'host.docker.internal' if IS_DOCKER else 'localhost'),
    'port': 5432,
    'database': 'wms_analysis',
    'user': 'postgres',
    'password': '123'
}

# Создание движка SQLAlchemy
conn_str = (
    f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)
engine = create_engine(conn_str)

# Проверка подключения
with engine.connect() as conn:
    result = conn.execute(text('SELECT 1'))
    print(f'Подключение к PostgreSQL установлено: {DB_CONFIG["host"]}:{DB_CONFIG["port"]}')

## Ячейка 3: Загрузка данных - Сотрудники

In [ ]:
# Загрузка признаков сотрудников
query_employee = "SELECT * FROM mart.feature_employee"
df_employee = pd.read_sql(query_employee, engine)

print(f'Загружено {len(df_employee)} строк из mart.feature_employee')
print(f'Столбцы: {list(df_employee.columns)}')
df_employee.head()

## Ячейка 4: Загрузка данных - Маршруты и Товары

In [ ]:
# Загрузка признаков маршрутов
query_route = "SELECT * FROM mart.feature_route"
df_route = pd.read_sql(query_route, engine)

print(f'Загружено {len(df_route)} строк из mart.feature_route')
print(f'Столбцы: {list(df_route.columns)}')

print('\n' + '='*60)

# Загрузка признаков товаров
query_item = "SELECT * FROM mart.feature_item"
df_item = pd.read_sql(query_item, engine)

print(f'Загружено {len(df_item)} строк из mart.feature_item')
print(f'Столбцы: {list(df_item.columns)}')

## Ячейка 5: Загрузка меток аномалий

In [ ]:
# Загрузка размеченных данных
query_labels = "SELECT employee_id, is_anomaly FROM mart.label_employee"
df_labels = pd.read_sql(query_labels, engine)

print(f'Загружено {len(df_labels)} меток аномалий')
print(f'\nРаспределение аномалий:')
print(df_labels['is_anomaly'].value_counts())
df_labels.head()

## Ячейка 6: Подготовка признаков для сотрудников

In [ ]:
# Признаки для анализа сотрудников
employee_feature_cols = ['scan_interval', 'scans_per_hour', 'time_since_last_scan']

# Подготовка признаков
X_employee = df_employee[employee_feature_cols].copy()
X_employee = X_employee.fillna(0)  # Заполняем пропуски нулями

print(f'Размер матрицы признаков: {X_employee.shape}')
print(f'\nСтатистика признаков:')
X_employee.describe()

## Ячейка 7: Подготовка признаков для маршрутов и товаров

In [ ]:
# Признаки для анализа маршрутов
route_feature_cols = ['error_rate', 'picked_per_operation', 'avg_waste_rate']

X_route = df_route[route_feature_cols].copy()
X_route = X_route.fillna(0)
X_route = X_route.replace([np.inf, -np.inf], 0)

print(f'Размер матрицы признаков (маршруты): {X_route.shape}')

print('\n' + '='*60)

# Признаки для анализа товаров (используем признаки с реальными данными)
item_feature_cols = ['total_picked', 'route_count', 'total_waste']

X_item = df_item[item_feature_cols].copy()
X_item = X_item.fillna(0)
X_item = X_item.replace([np.inf, -np.inf], 0)

print(f'\nРазмер матрицы признаков (товары): {X_item.shape}')

print('\nСтатистика признаков (маршруты):')
X_route.describe()

## Ячейка 8: Обучение Isolation Forest - Сотрудники

In [ ]:
# Масштабирование признаков сотрудников
scaler_employee = StandardScaler()
X_employee_scaled = scaler_employee.fit_transform(X_employee)

# Создание и обучение модели Isolation Forest
iso_forest_employee = IsolationForest(
    contamination=0.1,      # Ожидаем ~10% аномалий
    n_estimators=100,       # Количество деревьев
    random_state=42,        # Фиксированный seed для воспроизводимости
    n_jobs=-1               # Использовать все ядра CPU
)

# Предсказание аномалий
y_pred_employee = iso_forest_employee.fit_predict(X_employee_scaled)

# Получение оценок аномальности
anomaly_score_employee = iso_forest_employee.decision_function(X_employee_scaled)

# Преобразование предсказаний: -1 -> 1 (аномалия), 1 -> 0 (норма)
y_pred_employee_binary = (y_pred_employee == -1).astype(int)

print(f'Обучение модели для сотрудников завершено')
print(f'Обнаружено аномалий: {y_pred_employee_binary.sum()} из {len(y_pred_employee_binary)}')
print(f'Процент аномалий: {y_pred_employee_binary.mean()*100:.2f}%')

## Ячейка 9: Обучение Isolation Forest - Маршруты

In [ ]:
# Масштабирование признаков маршрутов
scaler_route = StandardScaler()
X_route_scaled = scaler_route.fit_transform(X_route)

# Создание и обучение модели Isolation Forest
iso_forest_route = IsolationForest(
    contamination=0.1,
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Предсказание аномалий
y_pred_route = iso_forest_route.fit_predict(X_route_scaled)

# Получение оценок аномальности
anomaly_score_route = iso_forest_route.decision_function(X_route_scaled)

# Преобразование предсказаний
y_pred_route_binary = (y_pred_route == -1).astype(int)

print(f'Обучение модели для маршрутов завершено')
print(f'Обнаружено аномалий: {y_pred_route_binary.sum()} из {len(y_pred_route_binary)}')
print(f'Процент аномалий: {y_pred_route_binary.mean()*100:.2f}%')

## Ячейка 10: Обучение Isolation Forest - Товары

In [ ]:
# Масштабирование признаков товаров
scaler_item = StandardScaler()
X_item_scaled = scaler_item.fit_transform(X_item)

# Создание и обучение модели Isolation Forest
iso_forest_item = IsolationForest(
    contamination=0.1,
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Предсказание аномалий
y_pred_item = iso_forest_item.fit_predict(X_item_scaled)

# Получение оценок аномальности
anomaly_score_item = iso_forest_item.decision_function(X_item_scaled)

# Преобразование предсказаний
y_pred_item_binary = (y_pred_item == -1).astype(int)

print(f'Обучение модели для товаров завершено')
print(f'Обнаружено аномалий: {y_pred_item_binary.sum()} из {len(y_pred_item_binary)}')
print(f'Процент аномалий: {y_pred_item_binary.mean()*100:.2f}%')

## Ячейка 11: Оценка модели - Сотрудники

In [ ]:
# Объединение данных с метками
df_employee_merged = df_employee.merge(df_labels, on='employee_id', how='left')
y_true_employee = df_employee_merged['is_anomaly'].fillna(0).astype(int).values

# Вычисление метрик
precision_emp = precision_score(y_true_employee, y_pred_employee_binary, zero_division=0)
recall_emp = recall_score(y_true_employee, y_pred_employee_binary, zero_division=0)
f1_emp = f1_score(y_true_employee, y_pred_employee_binary, zero_division=0)

print('МЕТРИКИ ДЛЯ СОТРУДНИКОВ')
print('='*40)
print(f'Precision: {precision_emp:.4f}')
print(f'Recall: {recall_emp:.4f}')
print(f'F1-Score: {f1_emp:.4f}')
print(f'\nОтчет по классификации:')
print(classification_report(y_true_employee, y_pred_employee_binary, 
                           target_names=['Норма', 'Аномалия']))

## Ячейка 12: Матрица ошибок - Сотрудники

In [ ]:
# Матрица ошибок для сотрудников
cm_employee = confusion_matrix(y_true_employee, y_pred_employee_binary)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_employee, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Норма', 'Аномалия'],
            yticklabels=['Норма', 'Аномалия'])
plt.title('Матрица ошибок - Сотрудники')
plt.ylabel('Истинные значения')
plt.xlabel('Предсказанные значения')
plt.tight_layout()
plt.show()

print(f'\nМатрица ошибок:\n{cm_employee}')

## Ячейка 13: Статистика аномалий для всех сущностей

In [ ]:
# Сводная статистика
stats = {
    'Сотрудники': {
        'Всего': len(y_pred_employee_binary),
        'Аномалий': y_pred_employee_binary.sum(),
        'Процент': y_pred_employee_binary.mean() * 100
    },
    'Маршруты': {
        'Всего': len(y_pred_route_binary),
        'Аномалий': y_pred_route_binary.sum(),
        'Процент': y_pred_route_binary.mean() * 100
    },
    'Товары': {
        'Всего': len(y_pred_item_binary),
        'Аномалий': y_pred_item_binary.sum(),
        'Процент': y_pred_item_binary.mean() * 100
    }
}

df_stats = pd.DataFrame(stats).T
print('СВОДНАЯ СТАТИСТИКА АНОМАЛИЙ')
print('='*50)
print(df_stats.to_string())

## Ячейка 14: Визуализация распределения аномалий

In [ ]:
# Распределение оценок аномальности для сотрудников
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Сотрудники
axes[0].hist(anomaly_score_employee, bins=50, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Граница аномалий')
axes[0].set_title('Распределение оценок - Сотрудники')
axes[0].set_xlabel('Оценка аномальности')
axes[0].set_ylabel('Частота')
axes[0].legend()

# Маршруты
axes[1].hist(anomaly_score_route, bins=50, alpha=0.7, color='coral', edgecolor='black')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Граница аномалий')
axes[1].set_title('Распределение оценок - Маршруты')
axes[1].set_xlabel('Оценка аномальности')
axes[1].set_ylabel('Частота')
axes[1].legend()

# Товары
axes[2].hist(anomaly_score_item, bins=50, alpha=0.7, color='mediumseagreen', edgecolor='black')
axes[2].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Граница аномалий')
axes[2].set_title('Распределение оценок - Товары')
axes[2].set_xlabel('Оценка аномальности')
axes[2].set_ylabel('Частота')
axes[2].legend()

plt.tight_layout()
plt.show()

## Ячейка 15: Scatter plots с выделением аномалий - Сотрудники

In [ ]:
# Scatter plot для сотрудников (scans_per_hour vs time_since_last_scan)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Нормальные точки
normal_mask = y_pred_employee_binary == 0
anomaly_mask = y_pred_employee_binary == 1

axes[0].scatter(df_employee.loc[normal_mask, 'scans_per_hour'],
               df_employee.loc[normal_mask, 'time_since_last_scan'],
               c='blue', alpha=0.5, label='Норма', s=20)
axes[0].scatter(df_employee.loc[anomaly_mask, 'scans_per_hour'],
               df_employee.loc[anomaly_mask, 'time_since_last_scan'],
               c='red', alpha=0.8, label='Аномалия', s=40, marker='x')
axes[0].set_xlabel('Сканирований в час')
axes[0].set_ylabel('Время с последнего сканирования')
axes[0].set_title('Сотрудники: Сканирования vs Время')
axes[0].legend()

# Оценки аномальности
scatter = axes[1].scatter(df_employee['scans_per_hour'],
                         df_employee['time_since_last_scan'],
                         c=anomaly_score_employee, cmap='RdYlGn_r',
                         alpha=0.6, s=30)
plt.colorbar(scatter, ax=axes[1], label='Оценка аномальности')
axes[1].set_xlabel('Сканирований в час')
axes[1].set_ylabel('Время с последнего сканирования')
axes[1].set_title('Сотрудники: Тепловая карта аномальности')

plt.tight_layout()
plt.show()

## Ячейка 16: Scatter plots - Маршруты и Товары

In [ ]:
# Scatter plot для маршрутов
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Маршруты
normal_mask_route = y_pred_route_binary == 0
anomaly_mask_route = y_pred_route_binary == 1

axes[0].scatter(df_route.loc[normal_mask_route, 'error_rate'],
               df_route.loc[normal_mask_route, 'picked_per_operation'],
               c='blue', alpha=0.3, label='Норма', s=10)
axes[0].scatter(df_route.loc[anomaly_mask_route, 'error_rate'],
               df_route.loc[anomaly_mask_route, 'picked_per_operation'],
               c='red', alpha=0.8, label='Аномалия', s=30, marker='x')
axes[0].set_xlabel('Процент ошибок')
axes[0].set_ylabel('Сборок за операцию')
axes[0].set_title('Маршруты: Ошибки vs Сборки')
axes[0].legend()

# Товары
normal_mask_item = y_pred_item_binary == 0
anomaly_mask_item = y_pred_item_binary == 1

axes[1].scatter(df_item.loc[normal_mask_item, 'total_picked'],
               df_item.loc[normal_mask_item, 'route_count'],
               c='blue', alpha=0.3, label='Норма', s=10)
axes[1].scatter(df_item.loc[anomaly_mask_item, 'total_picked'],
               df_item.loc[anomaly_mask_item, 'route_count'],
               c='red', alpha=0.8, label='Аномалия', s=30, marker='x')
axes[1].set_xlabel('Общее количество сборок')
axes[1].set_ylabel('Количество маршрутов')
axes[1].set_title('Товары: Сборки vs Маршруты')
axes[1].legend()

plt.tight_layout()
plt.show()

## Ячейка 17: Сохранение предсказаний в PostgreSQL

In [ ]:
def save_predictions(engine, entity_ids, entity_type, y_pred_binary, anomaly_score, features):
    """Сохранение предсказаний в PostgreSQL"""
    predictions_df = pd.DataFrame({
        'entity_id': entity_ids.astype(str),
        'entity_type': entity_type,
        'is_anomaly_predicted': y_pred_binary.astype(int),
        'anomaly_score': anomaly_score,
        'feature_1': features.iloc[:, 0] if len(features.columns) > 0 else 0,
        'feature_2': features.iloc[:, 1] if len(features.columns) > 1 else 0,
        'feature_3': features.iloc[:, 2] if len(features.columns) > 2 else 0,
        'created_at': datetime.now()
    })
    
    predictions_df.to_sql(
        'model_isolation_forest_predictions',
        engine,
        schema='mart',
        if_exists='append',
        index=False
    )
    print(f'Сохранено {len(predictions_df)} предсказаний для {entity_type}')

# Сохранение предсказаний для сотрудников
save_predictions(engine, df_employee['employee_id'], 'employee', 
                y_pred_employee_binary, anomaly_score_employee, X_employee)

# Сохранение предсказаний для маршрутов
save_predictions(engine, df_route['picking_route_id'], 'route',
                y_pred_route_binary, anomaly_score_route, X_route)

# Сохранение предсказаний для товаров
save_predictions(engine, df_item['item_id'], 'item',
                y_pred_item_binary, anomaly_score_item, X_item)

## Ячейка 18: Сохранение метрик в PostgreSQL

In [ ]:
def save_metrics(engine, entity_type, precision, recall, f1, anomaly_count, total_count):
    """Сохранение метрик в PostgreSQL"""
    metrics_df = pd.DataFrame([{
        'model_name': 'IsolationForest',
        'entity_type': entity_type,
        'precision_score': precision,
        'recall_score': recall,
        'f1_score': f1,
        'anomaly_count': anomaly_count,
        'total_count': total_count,
        'created_at': datetime.now()
    }])
    
    metrics_df.to_sql(
        'model_metrics',
        engine,
        schema='mart',
        if_exists='append',
        index=False
    )
    print(f'Сохранены метрики для {entity_type}')

# Сохранение метрик для сотрудников (с метками)
save_metrics(engine, 'employee', precision_emp, recall_emp, f1_emp,
            y_pred_employee_binary.sum(), len(y_pred_employee_binary))

# Сохранение метрик для маршрутов (без меток)
save_metrics(engine, 'route', 0.0, 0.0, 0.0,
            y_pred_route_binary.sum(), len(y_pred_route_binary))

# Сохранение метрик для товаров (без меток)
save_metrics(engine, 'item', 0.0, 0.0, 0.0,
            y_pred_item_binary.sum(), len(y_pred_item_binary))

## Ячейка 19: Итоговая сводка

In [ ]:
# Итоговая сводка
print('='*70)
print('ИТОГОВАЯ СВОДКА ОБУЧЕНИЯ ISOLATION FOREST')
print('='*70)
print(f'Дата обучения: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'\nПараметры модели:')
print(f'  - contamination: 0.1')
print(f'  - n_estimators: 100')
print(f'  - random_state: 42')

print(f'\nРезультаты по сущностям:')
print('-'*70)

# Таблица результатов
results_summary = []

for entity_type, y_pred_binary, y_true, n_total in [
    ('Сотрудники', y_pred_employee_binary, y_true_employee, len(y_pred_employee_binary)),
    ('Маршруты', y_pred_route_binary, None, len(y_pred_route_binary)),
    ('Товары', y_pred_item_binary, None, len(y_pred_item_binary))
]:
    anomaly_count = y_pred_binary.sum()
    anomaly_pct = y_pred_binary.mean() * 100
    
    if y_true is not None:
        p = precision_score(y_true, y_pred_binary, zero_division=0)
        r = recall_score(y_true, y_pred_binary, zero_division=0)
        f = f1_score(y_true, y_pred_binary, zero_division=0)
        results_summary.append({
            'Сущность': entity_type,
            'Всего': n_total,
            'Аномалий': anomaly_count,
            '% Аномалий': f'{anomaly_pct:.2f}%',
            'Precision': f'{p:.4f}',
            'Recall': f'{r:.4f}',
            'F1': f'{f:.4f}'
        })
    else:
        results_summary.append({
            'Сущность': entity_type,
            'Всего': n_total,
            'Аномалий': anomaly_count,
            '% Аномалий': f'{anomaly_pct:.2f}%',
            'Precision': 'N/A',
            'Recall': 'N/A',
            'F1': 'N/A'
        })

df_results = pd.DataFrame(results_summary)
print(df_results.to_string(index=False))

print(f'\nДанные сохранены в таблицы:')
print(f'  - mart.model_isolation_forest_predictions')
print(f'  - mart.model_metrics')
print('\n' + '='*70)